<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/AirlinesProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing libraries**

In [ ]:
#@title Dataset Link: https://www.kaggle.com/datasets/sjleshrac/airlines-customer-satisfaction

In [ ]:
# Importing necessary libraries
# Run once if needed: %pip install pandas numpy matplotlib seaborn scikit-learn kagglehub
import os
import numpy as np  # Used for numerical operations
import pandas as pd  # Used for data manipulation and analysis
import matplotlib.pyplot as plt  # Used for data visualization
import seaborn as sns  # Used for advanced data visualization
from sklearn.model_selection import train_test_split
import kagglehub

RANDOM_STATE = 42
path = kagglehub.dataset_download("sjleshrac/airlines-customer-satisfaction")

# **Loading the Data and split it**

In [ ]:
# Load the dataset explicitly rather than choosing an arbitrary CSV.
from pathlib import Path
csv_files = sorted(Path(path).glob("*.csv"))
if len(csv_files) != 1:
    raise ValueError(f"Expected one dataset CSV, found: {csv_files}")
data_file = csv_files[0]
data = pd.read_csv(data_file)  # Entire Dataset

# Separate features and labels
x = data.drop(columns=['satisfaction']) # The entire dataset except the target "satisfaction"
y = data['satisfaction'] # The target ONLY --> "satisfaction"

# Split the data into training, development, and test sets
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
x_dev, x_test, y_dev, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp)

# Output data split sizes
print(f"Training set: {x_train.shape}, {y_train.shape}")
print(f"Development set: {x_dev.shape}, {y_dev.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")

# Optional: Save the splits to separate files
output_dir = 'data_splits'
os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist

x_train.to_csv(os.path.join(output_dir, 'x_train.csv'), index=False)
y_train.to_csv(os.path.join(output_dir, 'y_train.csv'), index=False)
x_dev.to_csv(os.path.join(output_dir, 'x_dev.csv'), index=False)
y_dev.to_csv(os.path.join(output_dir, 'y_dev.csv'), index=False)
x_test.to_csv(os.path.join(output_dir, 'x_test.csv'), index=False)
y_test.to_csv(os.path.join(output_dir, 'y_test.csv'), index=False)

print(f"Data splits saved to '{output_dir}' directory.")

In [ ]:
print(data.shape)

In [ ]:
data.columns  # shows the columns in the dataset

In [ ]:
#@title .loc vs .iloc
"""
 (both start and end labels are included, unlike .iloc, which is exclusive of the end index).
 returns a subset of the DataFrame containing the rows in the specified range.


 Feature	              .loc	            .iloc
Based on:	           Labels (names) |	  Integer positions

End index behavior:	 Inclusive	    |    Exclusive

Works with:	       Strings, numbers |  Only numbers
                            slices.

Example:	          df.loc['a':'c'] |	df.iloc[0:3]


1️⃣ .loc (Label-Based Indexing)
        -Used to select rows and columns based on labels.
        -It includes the ending index.
        -Works with row labels (not necessarily integers).
        -Syntax: data.loc[row_label, column_label]


2️⃣ .iloc (Integer-Based Indexing)
        -Used to select rows and columns based on integer positions.
        -It excludes the ending index.
        -Only works with numerical positions.
        -Syntax: data.iloc[row_index, column_index]

#Summary
  -Label = The name assigned to a row (index) or column.
  -.loc[] selects rows/columns by label.
  -.iloc[] selects rows/columns by position.
"""
data.loc[0:4, 'satisfaction']  # selects data based on rows and column labels rather than numerical indices.
data.loc[:, :] # Selects all rows and columns
data.loc[:, 'Gender':'Seat comfort'] # Selects all rows and columns from 'Gender' to 'Seat comfort

In [ ]:
"""
When I retrieve a column as a list, it will return it as DataFrame
"""
desc = data.describe() # shows numerical columns only
#print(desc,'\n \n \n')
print(desc.loc['max'],'\n \n \n ')
print(data['Age'].count())  # Age here is a Pandas series
print(data[['Age']].count()) # Age here is Pandas Dataframe

# **Visualization of categorical variables**

In [ ]:
categorical_columns = x_train.select_dtypes(include=['object', 'category']).columns

plt.figure(figsize=(12, 8))
for i, col in enumerate(categorical_columns, 1):
    plt.subplot(3, 2, i)
    sns.countplot(y=x_train[col], order=x_train[col].value_counts().index, palette='magma')
    plt.title(f"{col} distribution")
    plt.tight_layout()
plt.show()
"""
Using enumerate makes code cleaner, more readable, and reduces the need for manual index tracking.
It's especially useful for working with loops that require both values and indices,
like when handling lists, subplots, file processing, or GUIs. 🚀
"""


# **Visualizing the target variable to check if there is imbalance**

In [ ]:
# Use training labels only when exploring the target distribution.
satisfaction = y_train.value_counts()
plt.figure(figsize=(8, 8))
plt.pie(satisfaction, labels=satisfaction.index, autopct='%1.1f%%')
plt.title('Satisfaction (training set)')
plt.show()

# **Step 1: Handle missing values**

In [ ]:
#@title Missing value check
missing_values = x_train.isnull().sum()
print(missing_values)

In [ ]:
# Libraries learn missing-value replacements from training data only.
# Keep x_train/x_dev/x_test raw; each model pipeline handles preprocessing.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone # Used to create a fresh copy of a model or pipeline, ensuring that the original remains unchanged.

numerical_columns = x_train.select_dtypes(include='number').columns.tolist()
categorical_columns = x_train.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numerical_columns),
    ('categorical', categorical_pipeline, categorical_columns),
])
# Median is a simple starting choice for the long-tailed delay values.
# Pipeline.fit learns these steps on training rows; predict only transforms.

# **Step 2: Data Encoding**

In [ ]:
# Replace 'satisfied' with 1 and 'dissatisfied' with 0 in the target variables.
label_mapping = {'satisfied': 1, 'dissatisfied': 0}
for labels in (y_train, y_dev, y_test):
    if not labels.isin(label_mapping).all():
        raise ValueError('Unexpected satisfaction label.')
y_train = y_train.map(label_mapping).astype(int)
y_dev = y_dev.map(label_mapping).astype(int)
y_test = y_test.map(label_mapping).astype(int)

In [ ]:
# One-hot encoding is now inside preprocessor.
# It preserves numerical values and produces the same feature columns for every split.
# Full one-hot encoding also avoids giving KNN an asymmetric dropped category.

In [ ]:
print(x_train.head())  # Raw features; preprocessing happens inside each model.

In [ ]:
# checking for target data types
print(y_train.dtypes)

# **Step 3: Feature Normalization**

In [ ]:
# StandardScaler replaces the hand-written min-max function.
# This handles constant columns safely and is a useful default for linear models/KNN.
# Trees do not require scaling; their model pipeline below uses imputation only.

In [ ]:
# Prepared copies are ONLY for the manual logistic-regression learning section.
# The library models below use raw features and their own complete pipelines.
manual_preprocessor = clone(preprocessor)
x_train_prepared = pd.DataFrame(
    manual_preprocessor.fit_transform(x_train), index=x_train.index)
x_dev_prepared = pd.DataFrame(
    manual_preprocessor.transform(x_dev), index=x_dev.index)
# Do not transform or inspect the test set during training/model comparison.

# **Our Model Functions**

In [ ]:
# Activation function
def sigmoid(z):  # It maps any real-valued number (output of the linear model) into a value between 0 and 1
  return np.exp(-np.logaddexp(0, -np.asarray(z)))  # Stable sigmoid


def predict(X, weights, bias):  # Output: Predicted probabilities for each data point.
  logistic_Regression = np.dot(X, weights) + bias
  return sigmoid(logistic_Regression)


def compute_loss(y_true, y_pred): # evaluates how well the predictions match the actual values.
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)  # Avoid log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# **Logistic Regression using miniBatch-gradient descent**
Implemented manually

In [ ]:
#@title Mini-batch gradient descent function
def mini_batch(x_train, y_train, x_dev, y_dev, learning_rate, num_iterations ,
               batch_size,
               lambda_reg ):

  num_samples, num_features = x_train.shape # Dimensions of training data
  weights = np.zeros(num_features)  # Initializes the model's weights as a zero vector
  """
  The number of weights should be equal to the number of features
  because each feature in the input data needs a corresponding weight in the linear model.

  If x_train has a shape of (num_samples, num_features)
  ,the weight vector must have a shape of (num_features,) so that the dot product X @ w works correctly.

  If the number of weights were different from the number of features
  ,the multiplication would be mathematically invalid.

  If we had more or fewer weights than features
  ,we wouldn't be able to correctly model the relationship between inputs and outputs.
  """
  bias = 0

# lists to store the accuracies for each data set
  train_acc = []
  dev_acc = []
  rng = np.random.default_rng(RANDOM_STATE)

# lists to store the loss for training and development data sets
  train_loss = []
  dev_loss = []

  for i in range(num_iterations): # for epochs

    batch_indices = rng.permutation(num_samples) # Randomly permutes the indices of the training data to shuffle it for each epoch.
    """
       ^  This means batch_indices contains all the indices of the training dataset but in a different, randomized order.

    The shuffled indices are used to rearrange x_train and y_train
    ,ensuring that the training data is shuffled at the beginning of each epoch.

    This helps improve the performance of mini-batch gradient descent
    by preventing the model from learning patterns based on the order of the data.

    """

    x_shuffled = x_train.iloc[batch_indices]
    y_shuffled = y_train.iloc[batch_indices]

    """
    x_shuffled is just x_train but with its rows rearranged according to the randomly shuffled indices in batch_indices.

    Why Use .iloc in Your Code?
      -It ensures that x_train and y_train are shuffled in the same way.
      -It reorders the data while keeping features and target values aligned.

    What Happens If You Only Shuffle x_train But Not y_train?
      -The features (x_train) will no longer correspond to the correct target labels (y_train).
      -This means that when your model tries to learn the relationship between features and labels, it's learning from incorrect pairings.
      As a result, the model won’t learn meaningful patterns and will perform poorly.
    """

    for start in range(0, num_samples, batch_size):     # for batch
      """
      In mini-batch gradient descent
     ,we update the model parameters (weights and bias) after processing each mini-batch
      """
      end = min(start + batch_size, num_samples)  # calculate the last index of the mini batch
      # Select the features and labels for the current batch.
      x_batch = x_shuffled.iloc[start:end]
      y_batch = y_shuffled.iloc[start:end]

      y_pred = predict(x_batch, weights, bias) # Predictions for the current mini-batch.
      """
      We compute y_pred for every mini-batch because it is needed for gradient updates in mini-batch training
      ,making the model more efficient and improving convergence.

      -w represents the weights before an update (pre-mini-batch in MBGD or pre-epoch in BGD).
      -After computing gradients, w is updated and used for the next iteration.
      -The updated w influences subsequent training steps, gradually improving model performance.
      """
      # weight and bias update
      weights -= learning_rate * ((1 / len(y_batch)) * np.dot(x_batch.T, (y_pred - y_batch)) + (lambda_reg / num_samples) * weights)
      bias -= learning_rate * (1 / len(y_batch)) * np.sum(y_pred - y_batch)  # the bias won't affect by L2 regularization

    # Record loss and accuracy AFTER the same epoch updates.
    train_probability = predict(x_train, weights, bias)
    dev_probability = predict(x_dev, weights, bias)
    penalty = lambda_reg / (2 * num_samples) * np.sum(weights ** 2)
    train_loss.append(compute_loss(y_train, train_probability) + penalty)
    dev_loss.append(compute_loss(y_dev, dev_probability))  # Predictive loss only
    train_acc.append(np.mean((train_probability >= 0.5) == y_train))
    dev_acc.append(np.mean((dev_probability >= 0.5) == y_dev))

  return weights, bias, train_acc, dev_acc, train_loss, dev_loss, train_probability

# **Calling the model**

In [ ]:
# Initializing hyperparameters (learning example; tune using development data).
learning_rate = 0.05
epochs = 30
batch_size = 256
lambda_reg = 1
weights, bias, train_acc, dev_acc, train_loss, dev_loss, pred = mini_batch(
    x_train_prepared, y_train, x_dev_prepared, y_dev,
    learning_rate, epochs, batch_size, lambda_reg,
)

# **Accuracy**

In [ ]:
print(f"Final Training Accuracy: {train_acc[-1]:.4f}")
print(f"Final Development Accuracy: {dev_acc[-1]:.4f}")
plt.plot(train_loss, label='Training loss (including L2)')
plt.plot(dev_loss, label='Development loss')
plt.xlabel('Epoch (zero-based)')
plt.ylabel('Log loss')
plt.legend()
plt.show()
# Inspect development loss to decide whether more epochs are helpful.

# **KNN Model**

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

results = []
fitted_models = {}
def evaluate_on_dev(name, estimator, preprocessing=preprocessor):
    # Clone ensures that each candidate learns its own preprocessing from training only.
    model = Pipeline([('preprocessor', clone(preprocessing)), ('model', estimator)])
    model.fit(x_train, y_train)
    predictions = model.predict(x_dev)
    results.append({'model': name,
                    'dev_accuracy': accuracy_score(y_dev, predictions),
                    'dev_macro_f1': f1_score(y_dev, predictions, average='macro')})
    fitted_models[name] = model
    print(name, results[-1])
    return model

evaluate_on_dev('Majority baseline', DummyClassifier(strategy='most_frequent'))
knn = evaluate_on_dev('KNN', KNeighborsClassifier(n_neighbors=15, n_jobs=-1))

In [ ]:
pd.DataFrame(results).sort_values('dev_macro_f1', ascending=False)

# **Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Trees need missing-value handling and encoding, but do not need scaling.
tree_preprocessor = clone(preprocessor)
tree_preprocessor.set_params(numeric__scaler='passthrough') # scaling is not necessary for decision trees, so we set the scaler to 'passthrough' to skip that step in the preprocessing pipeline.
tree = evaluate_on_dev('Decision tree', DecisionTreeClassifier(
    random_state=RANDOM_STATE, min_samples_leaf=5), tree_preprocessor)

In [ ]:
# The development score guides model selection; test data stays untouched.

# **Notes**

In [ ]:
"""
DataFrame.count: Count number of non-NA/null observations.
DataFrame.max: Maximum of the values in the object.
DataFrame.min: Minimum of the values in the object.
DataFrame.mean: Mean of the values.
DataFrame.std: Standard deviation of the observations.
DataFrame.select_dtypes: Subset of a DataFrame including/excluding
columns based on their dtype.


- In Forward Fill with axis [zero], the Nan values are filled based on the previous row.
- In Backward Fill with axis [zero], the Nan values are filled based on the next row.

- In Forward Fill with axis [one], the Nan values are filled based on the previous column.
- In Backward Fill with axis [one], the Nan values are filled based on the next column.

df.dropna(axis = 0 or 1)  will drop the Nan values in the specified axis
    """

In [ ]:
print(data.index)
range(data.shape[0])
df_temp = data.loc[::2, :]
print(df_temp.index)
print(range(df_temp.shape[0]))


In [ ]:
data.iloc[: , :7]
print(type('Departure/Arrival time convenient'))

In [ ]:
data[data.loc[:,'Departure/Arrival time convenient'] > data.loc[:, 'Departure/Arrival time convenient'].mean()]

In [ ]:
# data.fillna(value=(The values), limit=) The limit decides how many Nan values to replace in each column

In [ ]:
transposed_describe = data.describe().T
print(transposed_describe)

print(transposed_describe.loc["Age","mean"])

In [ ]:
# Scaling is already included in the model pipelines above.
# Do not fit another scaler or attach original column names to encoded data.
# To inspect prepared feature names:
print(manual_preprocessor.get_feature_names_out())

# **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
model = evaluate_on_dev('Logistic regression', LogisticRegression(max_iter=1000))

In [ ]:
# Use each named pipeline for predictions to avoid mixing predictions from different models.

# **Support Vector Machine**

In [ ]:
from sklearn.svm import LinearSVC
# LinearSVC is a practical linear SVM choice for this dataset size.
svm_model = evaluate_on_dev('Linear SVM', LinearSVC(
    C=0.1, dual=False, max_iter=5000, random_state=RANDOM_STATE))

"""
#Large C (e.g., 10, 100, 1000)
  -The model tries to classify every training data point correctly.
  -It results in a more complex decision boundary (low bias, high variance).
  -This increases the risk of overfitting.
#Small C (e.g., 0.1, 0.01, 0.001)
  -The model allows some misclassifications to keep the decision boundary smoother.
  -It prioritizes a larger margin over perfect classification.
  -This increases the risk of underfitting but improves generalization.
"""

## An ensemble candidate
Compare models on development macro F1 (equal weight for each class). These are starting settings, not an exhaustive search. If you tune further, use development data or cross-validation with the entire pipeline. Do not repeatedly check the final test score.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = evaluate_on_dev('Random forest', RandomForestClassifier(
    n_estimators=150, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    tree_preprocessor)
comparison = pd.DataFrame(results).sort_values('dev_macro_f1', ascending=False)
display(comparison)

## Final evaluation — run after model selection is finished
Choose the winner using development macro F1, refit its full pipeline on training + development data, and evaluate once on the untouched test set. Class 0 is dissatisfied, so its recall tells us how many dissatisfied passengers we find. Changing models after seeing these results makes this test set part of model development.

In [ ]:
best_name = comparison.iloc[0]['model']
final_model = clone(fitted_models[best_name])
final_model.fit(pd.concat([x_train, x_dev]), pd.concat([y_train, y_dev]))
final_predictions = final_model.predict(x_test)
print(f'Selected model: {best_name}')
print(f'Test accuracy: {accuracy_score(y_test, final_predictions):.4f}')
print(classification_report(y_test, final_predictions, labels=[0, 1],
      target_names=['dissatisfied', 'satisfied'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, final_predictions,
    labels=[0, 1], display_labels=['dissatisfied', 'satisfied'])
plt.show()

## Practical checks before using this model
- This workflow assumes independent passenger records. Check for duplicated records or repeated passengers before trusting a random split; group related records if identifiers are available.
- These features describe a completed flight. For pre-flight predictions, remove information unavailable at prediction time.
- Confirm the meaning of rating 0 with the dataset documentation; it may represent a special response.
- This test set was used in earlier notebook experiments. For a strictly fresh performance estimate, use a new external holdout.
- Save the complete final_model pipeline if you later need predictions on new raw rows.

## Original learning notes
The earlier explanations below are preserved for reference. They describe the original implementation; the executable workflow above uses library preprocessing instead.

<details>
<summary>Original cell 4: comments and implementation for reference</summary>

```python
# Importing necessary libraries
import os
import pandas as pd  # Used for data manipulation and analysis
import matplotlib.pyplot as plt  # Used for data visualization
import numpy as np  # Used for numerical operations
import seaborn as sns  # Used for advanced data visualization
from sklearn.model_selection import train_test_split  # Used for splitting data into training and testing sets
import kagglehub
path = kagglehub.dataset_download("sjleshrac/airlines-customer-satisfaction")
```

</details>

<details>
<summary>Original cell 6: comments and implementation for reference</summary>

```python
# Example: Load dataset from a file
file_list = os.listdir(path)
for file in file_list:
  if file.endswith('.csv'):
    data_file = os.path.join(path, file)
    break
# Ensure the file exists
#if not os.path.exists(data_file):
 #   raise FileNotFoundError(f"The file {data_file} does not exist.")

# Load the dataset
data = pd.read_csv(data_file) # Entire Dataset

# Separate features and labels
x = data.drop(columns=['satisfaction']) # The entire dataset except the target "satisfaction"
y = data['satisfaction'] # The target ONLY --> "satisfaction"

# Split the data into training, development, and test sets
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42)
x_dev, x_test, y_dev, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

# Output data split sizes
print(f"Training set: {x_train.shape}, {y_train.shape}")
print(f"Development set: {x_dev.shape}, {y_dev.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")

# Optional: Save the splits to separate files
output_dir = 'data_splits'
os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist

x_train.to_csv(os.path.join(output_dir, 'x_train.csv'), index=False)
y_train.to_csv(os.path.join(output_dir, 'y_train.csv'), index=False)
x_dev.to_csv(os.path.join(output_dir, 'x_dev.csv'), index=False)
y_dev.to_csv(os.path.join(output_dir, 'y_dev.csv'), index=False)
x_test.to_csv(os.path.join(output_dir, 'x_test.csv'), index=False)
y_test.to_csv(os.path.join(output_dir, 'y_test.csv'), index=False)

print(f"Data splits saved to '{output_dir}' directory.")
```

</details>

<details>
<summary>Original cell 14: comments and implementation for reference</summary>

```python
import matplotlib.pyplot as plt

# Generate the satisfaction data
satisfaction = data['satisfaction'].value_counts()

# Plot the pie chart
plt.figure(figsize=(8, 8))  # Adjust the figure size as needed
plt.pie(
    satisfaction,
    labels=['dissatisfied (0)', 'satisfied (1)'],
    autopct='%1.1f%%'
)
plt.title('Satisfaction')
plt.show()
```

</details>

<details>
<summary>Original cell 17: comments and implementation for reference</summary>

```python
#@title handle missing values
def handle_missing_value(df):
    """
    This function handles missing values in a DataFrame.

    It fills numerical columns with the mean and categorical columns with the mode.

    Args:
        df (pandas.DataFrame): The DataFrame to handle missing values in.

    Returns:
        pandas.DataFrame: The DataFrame with missing values filled.
    """
    # Get numerical columns
    numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns
    # Iterate through numerical columns and fill missing values with the mean
    for col in numerical_columns:
        if df[col].isnull().any(): # Checks if there is a missing value in each column
            df[col].fillna(value=df[f'{col}'].mean(), inplace=True)  # Fill with mean

    # Get categorical columns
    categorical_columns = df.select_dtypes(include=['object']).columns
    # Iterate through categorical columns and fill missing values with the mode
    for col in categorical_columns:
        if df[col].isnull().any():  # Check if column has any missing values
            df[col].fillna(value=df[f'{col}'].mode(), inplace=True)  # Fill with mode

    return df  # Return the modified DataFrame

# Apply the function to the training, development, and testing datasets
x_train = handle_missing_value(x_train)
x_dev = handle_missing_value(x_dev)
x_test = handle_missing_value(x_test)
```

</details>

<details>
<summary>Original cell 19: comments and implementation for reference</summary>

```python
#@title Replace 'satisfied' with 1 and 'dissatisfied' with 0 in the target variables
y_train.replace({'satisfied': 1, 'dissatisfied': 0}, inplace=True)
y_dev.replace({'satisfied': 1, 'dissatisfied': 0}, inplace=True)
y_test.replace({'satisfied': 1, 'dissatisfied': 0}, inplace=True)
```

</details>

<details>
<summary>Original cell 20: comments and implementation for reference</summary>

```python
#@title One-Hot Encoding
def Data_Encoding(df):
  """
  Performs one-hot encoding on categorical columns in a DataFrame.

  Args:
      df (pandas.DataFrame): The DataFrame containing categorical columns.

  Returns:
      pandas.DataFrame: The DataFrame with categorical columns one-hot encoded.
  """
  # Identify categorical columns
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns

  # Perform one-hot encoding using pandas' get_dummies
  dummies = pd.get_dummies(df, columns=categorical_columns, drop_first=True).astype(int)

  return dummies  # Return the DataFrame with one-hot encoded columns

# Apply one-hot encoding to the training, development, and testing datasets
x_train = Data_Encoding(x_train)
x_dev = Data_Encoding(x_dev)
x_test = Data_Encoding(x_test)
```

</details>

<details>
<summary>Original cell 24: comments and implementation for reference</summary>

```python
#@title Min-Max Normalization
def minmax_normalize(df, min_values=None, max_values=None):
    """
    Performs min-max normalization on a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to normalize.
        min_values (pandas.Series, optional): Pre-calculated minimum values for each column. Defaults to None.
        max_values (pandas.Series, optional): Pre-calculated maximum values for each column. Defaults to None.

    Returns:
        tuple: A tuple containing the normalized DataFrame, minimum values, and maximum values.
    """
    # Use provided min and max values if given, otherwise calculate them from the DataFrame
    min_values = min_values if min_values is not None else df.min() # Contains the minimum value of each column in df.
    max_values = max_values if max_values is not None else df.max() # Contains the maximum value of each column in df.

    # Normalize the DataFrame using the min-max formula
    normalized_df = (df - min_values) / (max_values - min_values)

    return normalized_df, min_values, max_values  # Return normalized DataFrame, min_values(useful for applying the same transformation to other datasets), and max_values
```

</details>

<details>
<summary>Original cell 25: comments and implementation for reference</summary>

```python
#@title Apply min-max normalization to the datasets
# Normalize training data and store min/max values
x_train, min_value, max_value = minmax_normalize(x_train)
x_test, _, _ = minmax_normalize(x_test, min_values=min_value, max_values=max_value)
x_dev, _, _ = minmax_normalize(x_dev, min_values=min_value, max_values=max_value)

# both test/dev sets are normalized using the min/max value from the training set to ensure consistency and valid evaluation and to avoid data leakage.
```

</details>

<details>
<summary>Original cell 27: comments and implementation for reference</summary>

```python
# Activation function
def sigmoid(z):  # It maps any real-valued number (output of the linear model) into a value between 0 and 1
  return (1 / (1 + np.exp(-z)))


def predict(X, weights, bias):  # Output: Predicted probabilities for each data point.
  logistic_Regression = np.dot(X, weights) + bias
  return sigmoid(logistic_Regression)


def compute_loss(y_true, y_pred): # evaluates how well the predictions match the actual values.
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
```

</details>

<details>
<summary>Original cell 29: comments and implementation for reference</summary>

```python
#@title Mini-batch gradient descent function
def mini_batch(x_train, y_train, x_dev, y_dev , x_test, y_test, learning_rate, num_iterations ,
               batch_size,
               lambda_reg ):

  num_samples, num_features = x_train.shape # Dimensions of training data
  weights = np.zeros(num_features)  # Initializes the model's weights as a zero vector
  """
  The number of weights should be equal to the number of features
  because each feature in the input data needs a corresponding weight in the linear model.

  If x_train has a shape of (num_samples, num_features)
  ,the weight vector must have a shape of (num_features,) so that the dot product X @ w works correctly.

  If the number of weights were different from the number of features
  ,the multiplication would be mathematically invalid.

  If we had more or fewer weights than features
  ,we wouldn't be able to correctly model the relationship between inputs and outputs.
  """
  bias = 0

# lists to store the accuracies for each data set
  train_acc = []
  dev_acc = []
  test_acc = []

# lists to store the loss for training and development data sets
  train_loss = []
  dev_loss = []

  for i in range(num_iterations): # for epochs

    batch_indices = np.random.permutation(num_samples) # Randomly permutes the indices of the training data to shuffle it for each epoch.
    """
       ^  This means batch_indices contains all the indices of the training dataset but in a different, randomized order.

    The shuffled indices are used to rearrange x_train and y_train
    ,ensuring that the training data is shuffled at the beginning of each epoch.

    This helps improve the performance of mini-batch gradient descent
    by preventing the model from learning patterns based on the order of the data.

    """

    x_shuffled = x_train.iloc[batch_indices]
    y_shuffled = y_train.iloc[batch_indices]

    """
    x_shuffled is just x_train but with its rows rearranged according to the randomly shuffled indices in batch_indices.

    Why Use .iloc in Your Code?
      -It ensures that x_train and y_train are shuffled in the same way.
      -It reorders the data while keeping features and target values aligned.

    What Happens If You Only Shuffle x_train But Not y_train?
      -The features (x_train) will no longer correspond to the correct target labels (y_train).
      -This means that when your model tries to learn the relationship between features and labels, it's learning from incorrect pairings.
      As a result, the model won’t learn meaningful patterns and will perform poorly.
    """

    # Compute and store training loss
    y_train_pred = predict(x_train, weights, bias)  # Predictions on the entire training dataset using the current weights and bias.
    prediction_on_training = y_train_pred
    regularization_term = (lambda_reg / (2 * num_samples)) * np.sum(weights**2)  # L2 regularization term
    train_loss.append(compute_loss(y_train, y_train_pred))

    # Compute and store development loss
    y_dev_pred = predict(x_dev, weights, bias)
    regularization_term_dev = (lambda_reg / (2 * len(y_dev))) * np.sum(weights**2)
    dev_loss.append(compute_loss(y_dev, y_dev_pred))

    for start in range(0, num_samples, batch_size):     # for batch
      """
      In mini-batch gradient descent
     ,we update the model parameters (weights and bias) after processing each mini-batch
      """
      end = min(start + batch_size, num_samples)  # calculate the last index of the mini batch
      # Select the features and labels for the current batch.
      x_batch = x_shuffled[start:end]
      y_batch = y_shuffled[start:end]

      y_pred = predict(x_batch, weights, bias) # Predictions for the current mini-batch.
      """
      We compute y_pred for every mini-batch because it is needed for gradient updates in mini-batch training
      ,making the model more efficient and improving convergence.

      -w represents the weights before an update (pre-mini-batch in MBGD or pre-epoch in BGD).
      -After computing gradients, w is updated and used for the next iteration.
      -The updated w influences subsequent training steps, gradually improving model performance.
      """
      # weight and bias update
      weights -= learning_rate * ((1 / len(y_batch)) * np.dot(x_batch.T, (y_pred - y_batch)) + (lambda_reg / num_samples) * weights)
      bias -= learning_rate * (1 / len(y_batch)) * np.sum(y_pred - y_batch)  # the bias won't affect by L2 regularization

    #compute and store accuracies
    y_train_pred = predict(x_train, weights, bias) >= 0.5
    train_acc.append(np.mean(y_train_pred == y_train)) # Compares predicted labels with true labels and calculates the mean (accuracy).

    y_dev_pred = predict(x_dev, weights, bias) >= 0.5
    dev_acc.append(np.mean(y_dev_pred == y_dev))

    y_test_pred = predict(x_test, weights, bias) >= 0.5
    test_acc.append(np.mean(y_test_pred == y_test))

  return weights, bias, train_acc, dev_acc, test_acc, train_loss, dev_loss, prediction_on_training
```

</details>

<details>
<summary>Original cell 31: comments and implementation for reference</summary>

```python
#@title initializing hyperparameters
learning_rate = 0.1
epochs = 5
batch_size = 64
lambda_reg = 1

weights, bias, train_acc, dev_acc, test_acc, train_loss, dev_loss, pred = mini_batch(
        x_train, y_train, x_dev, y_dev, x_test, y_test,
        learning_rate, epochs, batch_size, lambda_reg
    )
print(y_train)
print(pred)
```

</details>

<details>
<summary>Original cell 33: comments and implementation for reference</summary>

```python
#@title Display final accuracy results
print(f"\nFinal Training Accuracy: {train_acc[-1]:.4f}") # print the last train_acc value
print(f"Final Development Accuracy: {dev_acc[-1]:.4f}") # print the last dev_acc value
print(f"Final Test Accuracy: {test_acc[-1]:.4f}") # print the last test_acc value
```

</details>

<details>
<summary>Original cell 35: comments and implementation for reference</summary>

```python
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier  # For classification
from sklearn.neighbors import KNeighborsRegressor   # For regression

knn = KNeighborsClassifier(n_neighbors=15) # defining the model
knn.fit(x_train, y_train) # fitting / training the model
y_pred = knn.predict(x_test) # prediction on the test set
y_pred
accuracy = accuracy_score(y_test, y_pred) # calculating the accuracy
print(f"Accuracy: {accuracy:.4f}") # printing the accuracy
```

</details>

<details>
<summary>Original cell 48: comments and implementation for reference</summary>

```python
#@title Data Scaling
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
"""
features_scaled is a NumPy array returned by scaler.fit_transform().
NumPy arrays don’t have column names, so without specifying column names, Pandas would assign default numerical indices

Why Use StandardScaler?
  -Standardizing features ensures that they are on the same scale, which helps machine learning models perform better.
  -Many ML algorithms, such as logistic regression, SVMs, and neural networks, are sensitive to the scale of input data.
"""


scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(x_train)
data_scaled = pd.DataFrame(data=features_scaled, columns=['satisfaction', 'Gender', 'Customer Type', 'Age', 'Type of Travel',
       'Class', 'Flight Distance', 'Seat comfort',
       'Departure/Arrival time convenient', 'Food and drink', 'Gate location',
       'Inflight wifi service', 'Inflight entertainment', 'Online support',
       'Ease of Online booking', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes']) # providing the column names beacuse the variable features_scaled contain the scaled values without column names
print(features_scaled, "/n /n /n /n ")
print(data_scaled)
```

</details>

<details>
<summary>Original cell 50: comments and implementation for reference</summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss

model = LogisticRegression()
model.fit(x_train, y_train)  # The model is being trained
loss = log_loss(y_test, model.predict_proba(x_test))
print(f"Log Loss: {loss:.4f}")
proba = model.predict_proba(x_test) # used to get the probability for each data point belonging to each class
print(proba)

y_model_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_model_pred)
print(f"Accuracy: {accuracy:.4f}")
```

</details>

<details>
<summary>Original cell 53: comments and implementation for reference</summary>

```python
from sklearn.svm import SVC

svm_model = SVC(kernel='linear',C=0.001) # C is a regularization parameter that controls the trade-off between maximizing the margin and minimizing classification errors.
svm_model.fit(x_train, y_train)
y_pred = svm_model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

"""
#Large C (e.g., 10, 100, 1000)
  -The model tries to classify every training data point correctly.
  -It results in a more complex decision boundary (low bias, high variance).
  -This increases the risk of overfitting.
#Small C (e.g., 0.1, 0.01, 0.001)
  -The model allows some misclassifications to keep the decision boundary smoother.
  -It prioritizes a larger margin over perfect classification.
  -This increases the risk of underfitting but improves generalization.
"""
```

</details>